# 深度全连接神经网络：猫咪图像分类 🐱

欢迎来到本次实验！我们将利用前面实现的积木函数，搭建一个真正的深度神经网络，用于识别图片中是否有猫咪。🐾

**完成本实验后，你将能够：**

- 构建并训练一个两层神经网络（LINEAR → ReLU → LINEAR → Sigmoid）
- 构建并训练一个 $L$ 层深度神经网络，并观察更深网络带来的性能提升
- 理解网络深度对分类准确率的影响

让我们开始吧！🚀

## 📋 目录

- [1 - 准备工作](#1)
- [2 - 加载数据集](#2)
- [3 - 模型架构](#3)
    - [3.1 - 两层神经网络架构](#3-1)
- [4 - 两层神经网络](#4)
    - [练习1 - two_layer_model](#ex-1)
    - [4.1 - 训练模型](#4-1)
- [5 - L层深度神经网络](#5)
    - [练习2 - L_layer_model](#ex-2)
    - [5.1 - 训练L层模型](#5-1)
- [6 - 结果分析](#6)
- [7 - 总结与反思](#7)

<a name="1"></a>
## 1 - 准备工作 🛠️

首先导入所需的库和工具函数。

In [ ]:
### v1.1

In [ ]:
import time
import numpy as np
import h5py
import matplotlib.pyplot as plt
import scipy
from PIL import Image
from scipy import ndimage
from dnn_app_utils_v3 import *
from public_tests import *

%matplotlib inline
plt.rcParams['figure.figsize'] = (5.0, 4.0)
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

%load_ext autoreload
%autoreload 2

np.random.seed(1)

<a name="2"></a>
## 2 - 加载数据集 📂

我们使用猫咪与非猫咪的图像分类数据集。数据集包含：

- 训练集：若干张标注为猫（1）或非猫（0）的图片
- 测试集：用于评估模型泛化能力的图片

每张图片大小为 $64 \times 64 \times 3$（RGB 彩色）。

In [ ]:
train_x_orig, train_y, test_x_orig, test_y, classes = load_data()

### 🔍 探索数据集

运行下方单元格，随机查看一张训练图片。可以修改 `index` 变量多看几张～

In [ ]:
# 查看数据集中的示例图片
index = 10
plt.imshow(train_x_orig[index])
print("y = " + str(train_y[0, index]) + "，这是一张" + classes[train_y[0, index]].decode("utf-8") + "的图片。")

In [ ]:
# 查看数据集基本信息
m_train = train_x_orig.shape[0]
num_px = train_x_orig.shape[1]
m_test = test_x_orig.shape[0]

print("训练样本数: " + str(m_train))
print("测试样本数: " + str(m_test))
print("图像尺寸: (" + str(num_px) + ", " + str(num_px) + ", 3)")
print("train_x_orig 形状: " + str(train_x_orig.shape))
print("train_y 形状: " + str(train_y.shape))
print("test_x_orig 形状: " + str(test_x_orig.shape))
print("test_y 形状: " + str(test_y.shape))

### 🔧 数据预处理

在送入神经网络之前，需要对图像做两步处理：

1. **展平**：将每张 $64\times64\times3$ 的图片拉成一个长向量（共 $12288$ 个特征）
2. **归一化**：将像素值从 $[0, 255]$ 缩放到 $[0, 1]$，加速梯度下降收敛

In [ ]:
# 展平并归一化图像
train_x_flatten = train_x_orig.reshape(train_x_orig.shape[0], -1).T
test_x_flatten = test_x_orig.reshape(test_x_orig.shape[0], -1).T

train_x = train_x_flatten / 255.
test_x = test_x_flatten / 255.

print("train_x 形状: " + str(train_x.shape))
print("test_x 形状: " + str(test_x.shape))

> 💡 **提示**：$12288 = 64 \times 64 \times 3$，即一张图片展平后的特征维度。

<a name="3"></a>
## 3 - 模型架构 🏗️

本实验将构建并比较两种网络：

| 模型 | 结构 | 参数规模 |
|------|------|---------|
| 两层网络 | LINEAR→ReLU→LINEAR→Sigmoid | 较少 |
| L层网络 | [LINEAR→ReLU]×(L-1)→LINEAR→Sigmoid | 较多 |

<a name="3-1"></a>
### 3.1 - 两层神经网络架构

**网络结构**：输入层 → 隐藏层（ReLU）→ 输出层（Sigmoid）

- 输入：$n_x = 12288$（图像特征数）
- 隐藏层：$n_h = 7$ 个神经元
- 输出：$n_y = 1$（二分类：猫/非猫）

**通用方法论**：
1. 初始化参数
2. 前向传播（计算预测值）
3. 计算损失
4. 反向传播（计算梯度）
5. 更新参数
6. 重复步骤2-5直到收敛

<a name="4"></a>
## 4 - 两层神经网络 🔬

<a name="ex-1"></a>
### 练习1 - two_layer_model

利用已实现的辅助函数，构建两层神经网络（LINEAR→ReLU→LINEAR→Sigmoid）。

**可用函数**：
```python
initialize_parameters(n_x, n_h, n_y)      # 初始化参数
linear_activation_forward(A_prev, W, b, activation)  # 前向传播
compute_cost(AL, Y)                         # 计算损失
linear_activation_backward(dA, cache, activation)    # 反向传播
update_parameters(parameters, grads, lr)   # 更新参数
```

In [ ]:
### 模型常数 ###
n_x = 12288     # num_px * num_px * 3
n_h = 7
n_y = 1
layers_dims = (n_x, n_h, n_y)
learning_rate = 0.0075

In [ ]:
def two_layer_model(X, Y, layers_dims, learning_rate=0.0075, num_iterations=3000, print_cost=False):
    """
    实现两层神经网络: 线性 -> ReLU -> 线性 -> Sigmoid
    
    参数:
    X -- 输入数据，形状 (n_x, 样本数)
    Y -- 标签向量，形状 (1, 样本数)，1表示猫，0表示非猫
    layers_dims -- 各层维度 (n_x, n_h, n_y)
    learning_rate -- 学习率
    num_iterations -- 梯度下降迭代次数
    print_cost -- 是否每100次迭代打印损失
    
    返回:
    parameters -- 训练好的参数
    costs -- 损失列表
    """
    np.random.seed(1)
    grads = {}
    costs = []
    m = X.shape[1]
    (n_x, n_h, n_y) = layers_dims
    
    # 初始化参数
    parameters = initialize_parameters(n_x, n_h, n_y)
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    
    # 梯度下降循环
    for i in range(0, num_iterations):
        # 前向传播: 线性 -> ReLU -> 线性 -> Sigmoid
        A1, cache1 = linear_activation_forward(X, W1, b1, activation="relu")
        A2, cache2 = linear_activation_forward(A1, W2, b2, activation="sigmoid")
        
        # 计算损失
        cost = compute_cost(A2, Y)
        
        # 初始化反向传播
        dA2 = -(np.divide(Y, A2) - np.divide(1 - Y, 1 - A2))
        
        # 反向传播
        dA1, dW2, db2 = linear_activation_backward(dA2, cache2, activation="sigmoid")
        dA0, dW1, db1 = linear_activation_backward(dA1, cache1, activation="relu")
        
        grads['dW1'] = dW1
        grads['db1'] = db1
        grads['dW2'] = dW2
        grads['db2'] = db2
        
        # 更新参数
        parameters = update_parameters(parameters, grads, learning_rate)
        W1 = parameters["W1"]
        b1 = parameters["b1"]
        W2 = parameters["W2"]
        b2 = parameters["b2"]
        
        if print_cost and i % 100 == 0 or i == num_iterations - 1:
            print("第 %i 次迭代后的损失: %f" % (i, np.squeeze(cost)))
        if i % 100 == 0 or i == num_iterations:
            costs.append(cost)
    
    return parameters, costs

In [ ]:
parameters, costs = two_layer_model(train_x, train_y, layers_dims = (n_x, n_h, n_y), num_iterations = 2, print_cost=False)
print("第一次迭代后的损失: " + str(costs[0]))
two_layer_model_test(two_layer_model)

**✅ 期望输出：**

```
第一次迭代后的损失约为 0.693
All tests passed!
```

<a name="4-1"></a>
### 4.1 - 训练模型 🏋️

通过测试后，运行下方代码训练 2500 次迭代。训练过程中损失应单调下降。

> ⏱️ 预计运行时间：约 2-5 分钟。

In [ ]:
parameters, costs = two_layer_model(train_x, train_y, layers_dims = (n_x, n_h, n_y), num_iterations = 2500, print_cost=True)
plot_costs(costs, learning_rate)

**✅ 期望输出（部分）：**

| 迭代次数 | 损失值 |
|---------|-------|
| 第 0 次 | 0.6930497356599888 |
| 第 100 次 | 0.6464320953428849 |
| ... | ... |
| 第 2499 次 | 0.04421498215868956 |

In [ ]:
predictions_train = predict(train_x, train_y, parameters)

**✅ 期望输出：**

```
准确率: 1.0
```

In [ ]:
predictions_test = predict(test_x, test_y, parameters)

**✅ 期望输出：**

```
准确率: 0.72
```

### 🎉 恭喜！

两层神经网络在测试集上达到了 **72%** 的准确率，比逻辑回归的 70% 有所提升！

接下来，让我们看看更深的网络能否做得更好 💪

<a name="5"></a>
## 5 - L层深度神经网络 🏛️

<a name="ex-2"></a>
### 练习2 - L_layer_model

利用已实现的函数构建 $L$ 层网络：$[\text{LINEAR}\to\text{ReLU}]\times(L-1)\to\text{LINEAR}\to\text{Sigmoid}$

**可用函数**：
```python
initialize_parameters_deep(layers_dims)    # 初始化参数
L_model_forward(X, parameters)             # 前向传播
compute_cost(AL, Y)                         # 计算损失
L_model_backward(AL, Y, caches)            # 反向传播
update_parameters(parameters, grads, lr)   # 更新参数
```

In [ ]:
### 常数 ###
layers_dims = [12288, 20, 7, 5, 1]  # 4层模型

In [ ]:
def L_layer_model(X, Y, layers_dims, learning_rate=0.0075, num_iterations=3000, print_cost=False):
    """
    实现L层深度神经网络: [线性->ReLU]*(L-1) -> 线性 -> Sigmoid
    
    参数:
    X -- 输入数据，形状 (n_x, 样本数)
    Y -- 标签向量，形状 (1, 样本数)
    layers_dims -- 各层维度列表 [输入维度, 隐层1, ..., 输出维度]
    learning_rate -- 学习率
    num_iterations -- 梯度下降迭代次数
    print_cost -- 是否每100次迭代打印损失
    
    返回:
    parameters -- 训练好的参数
    costs -- 损失列表
    """
    np.random.seed(1)
    costs = []
    
    # 初始化参数
    parameters = initialize_parameters_deep(layers_dims)
    
    # 梯度下降循环
    for i in range(0, num_iterations):
        # 前向传播
        AL, caches = L_model_forward(X, parameters)
        
        # 计算损失
        cost = compute_cost(AL, Y)
        
        # 反向传播
        grads = L_model_backward(AL, Y, caches)
        
        # 更新参数
        parameters = update_parameters(parameters, grads, learning_rate)
        
        if print_cost and i % 100 == 0 or i == num_iterations - 1:
            print("第 %i 次迭代后的损失: %f" % (i, np.squeeze(cost)))
        if i % 100 == 0 or i == num_iterations:
            costs.append(cost)
    
    return parameters, costs

In [ ]:
parameters, costs = L_layer_model(train_x, train_y, layers_dims, num_iterations = 1, print_cost = False)
print("第一次迭代后的损失: " + str(costs[0]))
L_layer_model_test(L_layer_model)

<a name="5-1"></a>
### 5.1 - 训练L层模型 🏋️

通过测试后，运行下方代码训练 4 层深度神经网络，迭代 2500 次。

> ⏱️ 预计运行时间：约 2-5 分钟。

In [ ]:
parameters, costs = L_layer_model(train_x, train_y, layers_dims, num_iterations = 2500, print_cost = True)

**✅ 期望输出（部分）：**

| 迭代次数 | 损失值 |
|---------|-------|
| 第 0 次 | 0.771749 |
| 第 100 次 | 0.672053 |
| ... | ... |
| 第 2499 次 | 0.088439 |

In [ ]:
pred_train = predict(train_x, train_y, parameters)

**✅ 期望输出：**

```
准确率: 0.9856
```

In [ ]:
pred_test = predict(test_x, test_y, parameters)

**✅ 期望输出：**

```
准确率: 0.8
```

### 🎉 太棒了！

4 层深度神经网络在测试集上达到了 **80%** 的准确率，比两层网络的 72% 提升显著！

这说明增加网络深度能有效提升模型的表达能力。在后续课程中，我们还将学习更多技巧来进一步提升性能。✨

<a name="6"></a>
## 6 - 结果分析 🔍

让我们看看模型错误分类了哪些图片，分析失败原因。

In [ ]:
print_mislabeled_images(classes, test_x, test_y, pred_test)

### 📊 常见错误类型分析

模型容易在以下情况下出错：

| 错误类型 | 描述 |
|---------|------|
| 姿势异常 | 猫咪处于不常见的姿势或角度 |
| 背景混淆 | 猫咪颜色与背景相近 |
| 特殊品种 | 罕见颜色或品种的猫咪 |
| 光线问题 | 过亮或过暗的照片 |
| 尺度变化 | 猫咪在图中占比过大或过小 |

> 💡 这些问题可通过数据增强、迁移学习等更高级技术来改善。

<a name="7"></a>
## 7 - 总结与反思 📝

### 本次实验学到了什么？

✅ 构建了两层神经网络（72% 准确率）

✅ 构建了4层深度神经网络（80% 准确率）

✅ 验证了更深的网络能提升分类性能

### 思考问题 🤔

1. 如果继续增加网络层数，准确率会一直提升吗？为什么？
2. 训练集准确率接近100%，但测试集只有80%，这说明了什么？
3. 学习率 `learning_rate=0.0075` 是如何影响训练过程的？

### 可选挑战 🌟

运行下方单元格，用自己的图片测试模型！

步骤：
1. 将图片上传至 `images/` 文件夹
2. 修改 `my_image` 为图片文件名
3. 设置 `my_label_y`（1=猫，0=非猫）
4. 运行并查看结果

In [ ]:
## 测试自己的图片（可选） ##
my_image = "my_image.jpg"  # 替换为你的图片文件名
my_label_y = [1]            # 真实类别：1=猫，0=非猫

fname = "images/" + my_image
image = np.array(Image.open(fname).resize((num_px, num_px)))
plt.imshow(image)
image = image / 255.
image = image.reshape((1, num_px * num_px * 3)).T

my_predicted_image = predict(image, my_label_y, parameters)

print("y = " + str(np.squeeze(my_predicted_image)) + "，模型预测这是一张\"" + classes[int(np.squeeze(my_predicted_image))].decode("utf-8") + "\"的图片。")